# 2. Designing a fair comparison

**Goal:** explain why splitting, pairing, and selection are part of the experiment itself. Strong software engineering makes these rules enforceable; statistics explains why they matter.

Everything here is synthetic. The identifiers and scores are invented, not NTU records or project results. Cells require only NumPy and perform no file, network, or cluster operations.

## Split actions according to the question

A random clip split asks whether a model recognizes more examples of actions it may already know. Pose Embed asks whether learned similarity transfers to **held-out actions**, given one clean reference per action. Sets must therefore be disjoint by action, not merely by filename.

During selection, fit on 80 actions and choose settings on 20 separate development actions. After selection, train fresh final heads on all 100 auxiliary actions. The final 20 novel actions remain outside training and selection. This toy uses **8 + 2 + 2** actions to make the relationship visible.

Development actions becoming final-training actions is deliberate: their information has already been used to choose a recipe. They are not subsequently presented as the untouched final test. Action separation also does not automatically imply performer separation.

In [1]:
import numpy as np

selection_train_actions = set(range(1, 9))
development_actions = {9, 10}
novel_actions = {11, 12}
final_train_actions = selection_train_actions | development_actions
assert selection_train_actions.isdisjoint(development_actions)
assert final_train_actions.isdisjoint(novel_actions)
assert len(selection_train_actions | development_actions | novel_actions) == 12
print('Selection training:', sorted(selection_train_actions))
print('Development selection:', sorted(development_actions))
print('Fresh final training:', sorted(final_train_actions))
print('Final evaluation only:', sorted(novel_actions))

Selection training: [1, 2, 3, 4, 5, 6, 7, 8]
Development selection: [9, 10]
Fresh final training: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Final evaluation only: [11, 12]


## A different camera is not necessarily a different performance

Suppose an anchor and query show the same person performing the same action at the same time, from different cameras. Their filenames differ, but the underlying performance is shared. Keeping both can make matching easier than matching a new performance.

The study identifies a performance by `(setup, performer, repetition, action)`; camera is deliberately excluded. Its primary query set excludes the gallery performance's synchronized views. The exact-official query set is also reported descriptively under its own fixed definition. Neither set can change after observing results.

In [2]:
# Tuple fields: setup, performer, repetition, action, camera.
anchor = (1, 2, 1, 9, 1)
candidates = [
    (1, 2, 1, 9, 2),  # Same performance as anchor, different camera.
    (1, 2, 2, 9, 1),  # Different repetition.
    (1, 3, 1, 9, 1),  # Different performer.
]
filename_only_queries = [row for row in candidates if row != anchor]

def performance(row):
    return row[:4]

primary_queries = [row for row in candidates if performance(row) != performance(anchor)]
assert len(filename_only_queries) == 3
assert len(primary_queries) == 2
assert all(performance(row) != performance(anchor) for row in primary_queries)
print('Removing only the identical record leaves:', len(filename_only_queries), 'queries')
print('Removing the whole anchor performance leaves:', len(primary_queries), 'queries')
print('The removed view was not an independent performance.')

Removing only the identical record leaves: 3 queries
Removing the whole anchor performance leaves: 2 queries
The removed view was not an independent performance.


## Pair sources of randomness before comparing methods

Initial weights and batches can change accuracy. With unrelated draws, a comparison contains both the recipe effect and extra random variation. The study pairs initialization and the **physical** batches within each seed, and uses the same realized query corruptions. Its fixed seeds are 7, 17, and 29.

For paired measurements $A_s,B_s$,
\[
\operatorname{Var}(A-B)=\operatorname{Var}(A)+\operatorname{Var}(B)-2\operatorname{Cov}(A,B).
\]
Shared easy or difficult conditions can produce positive covariance and reduce variability of the difference. Pairing does not guarantee a gain, equalize trained parameters, or create more than three training seeds. A common seed alone is not enough: actual starting weights and batch identities must agree before training.

The next cell deliberately constructs shared nuisance variation. Permuting the second method breaks seed matching without changing either method's mean.

In [3]:
seeds = np.array([7, 17, 29])
shared_variation = np.array([-0.03, 0.01, 0.02])
contrastive = 0.70 + shared_variation
contextual = 0.72 + shared_variation + np.array([-0.003, 0.002, 0.001])
paired_gain = contextual - contrastive
mismatched_gain = contextual[[2, 0, 1]] - contrastive
assert np.isclose(paired_gain.mean(), mismatched_gain.mean())
assert paired_gain.std(ddof=1) < mismatched_gain.std(ddof=1)
print('Paired seed gains (percentage points):', np.round(100 * paired_gain, 2))
print(f'Paired spread: {100 * paired_gain.std(ddof=1):.2f} percentage points')
print(f'Mismatched spread: {100 * mismatched_gain.std(ddof=1):.2f} percentage points')
print('This illustrates the design principle, not a measured project benefit.')

Paired seed gains (percentage points): [1.7 2.2 2.1]
Paired spread: 0.26 percentage points
Mismatched spread: 4.75 percentage points
This illustrates the design principle, not a measured project benefit.


## Give each recipe the same tuning opportunity

Fairness does not require the same winning learning rate. It requires the same **search budget and selection rule**. Each method gets three fixed rates, each run with all three seeds: $3\times3\times3=27$ selection runs. Every run uses 20 epochs. There is no best-epoch selection or grid expansion for an inconvenient winner.

The real rule ranks candidates by three-seed mean clean-development Top-1, then mean MRR, then lower learning rate. The toy scores have no ties, so Top-1 determines the winner. Pilots and corrupted scores cannot select a rate. A candidate missing a valid seed cannot compete using a partial average.

In [4]:
rates = (1e-4, 3e-4, 1e-3)
methods = ('contrastive', 'contextual', 'supcon')
# Rows are learning rates; columns are the same three fixed seeds.
candidate_scores = {
    'contrastive': np.array([0.68, 0.72, 0.69])[:, None] + shared_variation,
    'contextual': np.array([0.70, 0.73, 0.75])[:, None] + shared_variation,
    'supcon': np.array([0.71, 0.74, 0.72])[:, None] + shared_variation,
}
assert sum(scores.size for scores in candidate_scores.values()) == 27
for method in methods:
    scores = candidate_scores[method]
    assert scores.shape == (len(rates), len(seeds))
    means = scores.mean(axis=1)
    winner = int(np.argmax(means))
    print(f'{method:11s}: mean scores {np.round(means, 3)}; selected rate {rates[winner]:g}')
complete = np.array([True, False, True])
assert not complete.all()  # Partial candidates are ineligible, even when observed scores look good.
print('After selection: train 3 methods x 3 seeds = 9 fresh final heads.')

contrastive: mean scores [0.68 0.72 0.69]; selected rate 0.0003
contextual : mean scores [0.7  0.73 0.75]; selected rate 0.001
supcon     : mean scores [0.71 0.74 0.72]; selected rate 0.0003
After selection: train 3 methods x 3 seeds = 9 fresh final heads.


## What makes the test honest?

Freeze rules; check implementation; run complete pilots; verify resources; run selection; lock selected rates; train nine fresh heads; lock all nine; only then open the final test. Once final outcomes are visible, no new tuning, split, or favorable subset can enter the confirmatory result.

**Self-checks and answers**

1. **Why not choose settings on noisy development queries?** The frozen selection rule uses clean development only. Corruption scores would add an unplanned target and selection opportunity.
2. **Why not stop a poor method early?** It would receive a different budget. Poor accuracy is a valid outcome, not an execution failure.
3. **Why are microbatches not automatically one large contextual batch?** The contextual loss depends on which examples coexist. Accumulating gradients from different neighbor sets computes a different objective.
4. **Does this split prove performance on unseen people?** No. Its main separation is by action; the claim must match that design.
5. **Why new final heads?** Selection evaluates settings on held-out development actions; final training refits the selected settings on all 100 auxiliary actions with fresh paired starts.

See the [final plan](../../plan/research-plan.v3.md) for actual identities and [v3 protocol](../../docs/protocol/protocol-v3.md) for fixed rules.